In [1]:
!git clone https://github.com/keiranoluv/final_project
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'final_project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 38 (delta 8), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 18.64 KiB | 1.04 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1137/1137), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 353017 (delta 1064), reused 990 (delta 990), pack-reused 351880 (from 3)
Receiving objects: 100% (353017/353017), 1.87 GiB | 31.26 MiB/s, done.
Resolving deltas: 100% (279169/279169), done.


In [2]:
%cd /kaggle/working/PaddleOCR

!python -m pip install -q -r requirements.txt
!python -m pip install -q paddlepaddle-gpu==3.3.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  --no-deps

/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 932.5 kB/s eta 0:00:00


In [3]:
import csv
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2")
OUTPUT_ROOT = Path("/kaggle/working/mthv2_labels")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    src = DATASET_ROOT / f"{split}.tsv"
    dst = OUTPUT_ROOT / f"{split}.txt"

    count = 0

    with src.open("r", encoding="utf-8") as fin, \
         dst.open("w", encoding="utf-8") as fout:

        reader = csv.DictReader(fin, delimiter="\t")

        for row in reader:
            fout.write(f'{row["image_path"]}\t{row["text"]}\n')
            count += 1

    print(f"{split}: {count:,} samples -> {dst}")

train: 72,563 samples -> /kaggle/working/mthv2_labels/train.txt
val: 7,753 samples -> /kaggle/working/mthv2_labels/val.txt
test: 25,262 samples -> /kaggle/working/mthv2_labels/test.txt


In [4]:
TRAIN_CHARS = Path(
    "/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2/train_characters.txt"
)

PADDLE_DICT = Path(
    "/kaggle/working/PaddleOCR/ppocr/utils/dict/ppocrv6_dict.txt"
)

train_chars = {
    line.rstrip("\r\n")
    for line in TRAIN_CHARS.open("r", encoding="utf-8")
}

paddle_chars = {
    line.rstrip("\r\n")
    for line in PADDLE_DICT.open("r", encoding="utf-8")
}

# bỏ dòng rỗng nhưng GIỮ space nếu dataset thực sự có
train_chars.discard("")
paddle_chars.discard("")

missing = train_chars - paddle_chars
covered = train_chars & paddle_chars

print("=== Character coverage ===")
print(f"MTHv2 train unique chars : {len(train_chars):,}")
print(f"PP-OCRv6 dict chars      : {len(paddle_chars):,}")
print(f"Covered                  : {len(covered):,}")
print(f"Missing                  : {len(missing):,}")
print(f"Coverage                 : {len(covered) / len(train_chars) * 100:.4f}%")

=== Character coverage ===
MTHv2 train unique chars : 6,063
PP-OCRv6 dict chars      : 18,708
Covered                  : 5,278
Missing                  : 785
Coverage                 : 87.0526%


## B1 – Fine-tuning với dictionary mặc định của PP-OCRv6

B1 fine-tune mô hình `PP-OCRv6_medium_rec` trên tập MTHv2 nhưng vẫn giữ nguyên **character dictionary mặc định của PP-OCRv6**.

### Character coverage

Tập train MTHv2 có:

* **6,063** ký tự khác nhau.
* Dictionary mặc định của PP-OCRv6 sử dụng file `ppocrv6_dict.txt`.
* Character coverage cần được tính lại dựa trên dictionary này.
* Các ký tự trong tập train không xuất hiện trong `ppocrv6_dict.txt` được xem là **OOV (Out Of Vocabulary)**.

Xét theo số lần xuất hiện, cần thống kê:

* Tổng số ký tự trong tập train.
* Tổng số lần xuất hiện của các ký tự OOV.
* Tỷ lệ OOV theo số lần xuất hiện.

Các giá trị coverage của PP-OCRv5 như **5,277 / 6,063**, **87.04%**, **786 OOV** và **1.39%** không nên giữ lại, vì PP-OCRv6 sử dụng dictionary khác và cần được tính lại từ `ppocrv6_dict.txt`.

### Ảnh hưởng của OOV trong PaddleOCR

PaddleOCR sử dụng `character_dict_path` để ánh xạ từng ký tự trong ground truth sang chỉ số tương ứng trong vocabulary.

Với `PP-OCRv6_medium_rec`, dictionary mặc định được cấu hình là:

```text
ppocr/utils/dict/ppocrv6_dict.txt
```

Nếu một ký tự trong ground truth không tồn tại trong dictionary, ký tự đó không thể được biểu diễn bằng vocabulary hiện tại và sẽ không được sử dụng như một target character hợp lệ trong quá trình encode label.

Ví dụ:

```text
Ground truth gốc:
天地䖏玄黃

Nếu ký tự 䖏 không nằm trong dictionary:
天地玄黃
```

Do đó, trước khi fine-tune `PP-OCRv6_medium_rec` trên MTHv2, cần kiểm tra lại character coverage và OOV rate dựa trên `ppocrv6_dict.txt`.


In [5]:
!mkdir -p pretrained

!wget -O pretrained/PP-OCRv6_medium_rec_pretrained.pdparams \
  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams

--2026-09-04 14:22:31--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 232651336 (222M) [application/octet-stream]
Saving to: ‘pretrained/PP-OCRv6_medium_rec_pretrained.pdparams’

pretrained/PP-OCRv6 100%[===================>] 221.87M  6.69MB/s    in 50s     

2026-09-04 14:23:22 (4.45 MB/s) - ‘pretrained/PP-OCRv6_medium_rec_pretrained.pdparams’ saved [232651336/232651336]



In [6]:
%cd /kaggle/working/PaddleOCR

!python -m paddle.distributed.launch \
  --gpus "0,1" \
  tools/train.py \
  -c configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml \
  -o \
  Global.pretrained_model=./pretrained/PP-OCRv6_medium_rec_pretrained.pdparams \
  Global.epoch_num=20 \
  Global.save_model_dir=/kaggle/working/final_project/outputs/B1_v6_medium_20epochs_2gpu_bs64 \
  Global.eval_batch_step="[0,500]" \
  Train.loader.batch_size_per_card=64 \
  Train.sampler.first_bs=64 \
  Train.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Train.dataset.label_file_list='["/kaggle/working/mthv2_labels/train.txt"]' \
  Eval.dataset.data_dir=/kaggle/input/datasets/zephyrvn/nlp-project-data/MTHv2_processed/MTHv2 \
  Eval.dataset.label_file_list='["/kaggle/working/mthv2_labels/val.txt"]'

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
LAUNCH INFO 2026-09-04 14:23:29,700 -----------  Configuration  ----------------------
LAUNCH INFO 2026-09-04 14:23:29,700 auto_cluster_config: 0
LAUNCH INFO 2026-09-04 14:23:29,700 auto_parallel_config: None
LAUNCH INFO 2026-09-04 14:23:29,700 auto_tuner_json: None
LAUNCH INFO 2026-09-04 14:23:29,700 devices: 0,1
LAUNCH INFO 2026-09-04 14:23:29,700 elastic_level: -1
LAUNCH INFO 2026-09-04 14:23:29,701 elastic_timeout: 30
LAUNCH INFO 2026-09-04 14:23:29,701 enable_gpu_log: True
LAUNCH INFO 2026-09-04 14:23:29,701 gloo_port: 6767
LAUNCH INFO 2026-09-04 14:23:29,701 host: None
LAUNCH INFO 2026-09-04 14:23:29,701 ips: None
LAUNCH INFO 2026-09-04 